# Install Module
> MCP configuration installers for various AI tools

In [ ]:
#| default_exp install.__init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from typing import Optional, List

# Re-export from base (which re-exports from utils)
from nbdev_mcp.install.base import (
    InstallResult, InstallAction, MCPServerConfig,
    BaseInstaller, Installer,
    get_python_path, which, run_command,
    read_json_config, write_json_config, read_toml_config, write_toml_config,
    get_vscode_mcp_path, get_vscode_user_dir,
    get_claude_config_path, get_claude_config_dir,
    get_codex_config_path, get_codex_config_dir,
    get_ollama_config_path, get_ollama_config_dir,
    get_all_config_paths, get_mcp_config_path
)

from nbdev_mcp.install.claude import ClaudeInstaller, install_claude, uninstall_claude
from nbdev_mcp.install.codex import CodexInstaller, install_codex, uninstall_codex
from nbdev_mcp.install.vscode import VSCodeInstaller, install_vscode, uninstall_vscode
from nbdev_mcp.install.ollama import OllamaInstaller, install_ollama, uninstall_ollama

In [ ]:
#| export
PROVIDERS = ['claude', 'codex', 'vscode', 'ollama']
"""List of supported provider names."""

## High-Level Functions

In [ ]:
#| export
def install_all(
    python_path: Optional[str] = None,
    force: bool = False,
    dry_run: bool = False,
    providers: Optional[List[str]] = None
) -> List[InstallResult]:
    """Install nbdev-mcp configuration for all or specified providers.
    
    Parameters
    ----------
    python_path : str, optional
        Python interpreter path. Defaults to current.
    force : bool
        Overwrite existing configurations.
    dry_run : bool
        Show what would be done without making changes.
    providers : list[str], optional
        List of providers to install. Defaults to all.
    
    Returns
    -------
    list[InstallResult]
        Results for each provider.
    """
    if providers is None:
        providers = PROVIDERS
    
    server = MCPServerConfig.for_nbdev(python_path)
    results = []
    
    for provider in providers:
        installer = get_installer(provider)
        if installer:
            result = installer.install(server, force=force, dry_run=dry_run)
            results.append(result)
    
    return results


def uninstall_all(
    dry_run: bool = False,
    providers: Optional[List[str]] = None
) -> List[InstallResult]:
    """Uninstall nbdev-mcp configuration from all or specified providers.
    
    Parameters
    ----------
    dry_run : bool
        Show what would be done without making changes.
    providers : list[str], optional
        List of providers to uninstall. Defaults to all.
    
    Returns
    -------
    list[InstallResult]
        Results for each provider.
    """
    if providers is None:
        providers = PROVIDERS
    
    results = []
    
    for provider in providers:
        installer = get_installer(provider)
        if installer:
            result = installer.uninstall(dry_run=dry_run)
            results.append(result)
    
    return results

## Status Functions

In [ ]:
#| export
def check_status(providers: Optional[List[str]] = None) -> List[dict]:
    """Check installation status for all or specified providers.
    
    Parameters
    ----------
    providers : list[str], optional
        List of providers to check. Defaults to all.
    
    Returns
    -------
    list[dict]
        Status information for each provider.
    """
    if providers is None:
        providers = PROVIDERS
    
    statuses = []
    
    for provider in providers:
        installer = get_installer(provider)
        if installer:
            statuses.append(installer.status())
    
    return statuses


def print_status(providers: Optional[List[str]] = None) -> None:
    """Print installation status for all or specified providers.
    
    Parameters
    ----------
    providers : list[str], optional
        List of providers to check. Defaults to all.
    """
    statuses = check_status(providers)
    
    print("\nnbdev-mcp Installation Status")
    print("=" * 40)
    
    for status in statuses:
        provider = status['provider']
        configured = status['configured']
        exists = status['exists']
        path = status['path']
        
        icon = "[x]" if configured else "[ ]"
        note = "" if exists else " (config file not found)"
        
        print(f"{icon} {provider:10s} {path}{note}")
    
    print()

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def get_installer(provider: str) -> Optional[BaseInstaller]:
    """Get installer instance for a provider.
    
    Parameters
    ----------
    provider : str
        Provider name (claude, codex, vscode, ollama).
    
    Returns
    -------
    Optional[BaseInstaller]
        Installer instance or None if unknown provider.
    """
    installers = {
        'claude': ClaudeInstaller,
        'codex': CodexInstaller,
        'vscode': VSCodeInstaller,
        'ollama': OllamaInstaller,
    }
    cls = installers.get(provider.lower())
    return cls() if cls else None